The purpose of this notebook is to demonstrate how to perform streamflow evaluation using NOAA National Water Model retrospective predictions.

In [104]:
import s3fs
import pandas as pd
import xarray as xr
import geopandas as gpd
from datetime import datetime
from dask.distributed import Client

import hf_hydrodata as hf



import hydrodata_creds # local file containing my hydrodata credentials

In [105]:
# You need to register on https://hydrogen.princeton.edu/pin 
# and run the following with your registered information
# before you can use the hydrodata utilities
hf.register_api_pin(hydrodata_creds.username,
                    hydrodata_creds.pin)

In [106]:
# Define the time range of our analysis
start_date = '2018-10-01'
end_date = '2018-12-30'

Gather streamflow observations from HydroData.

In [217]:
obs_sites = hf.get_site_variables(datase='usgs_nwis',
                                  variable='streamflow',
                                  date_start=start_date,
                                  date_end=end_date,
                                  huc_id=['02030105'],
                                  grid = 'conus2',
                                  )  

In [218]:
obs_sites.head()

,site_id,site_name,site_type,agency,state,variable_name,units,dataset,variable,temporal_resolution,...,latitude,longitude,site_query_url,date_metadata_last_updated,tz_cd,doi,conus1_i,conus1_j,conus2_i,conus2_j
0,01396500,South Branch Raritan River near High Bridge NJ,stream gauge,USGS,NJ,Hourly average streamflow,m3/s,usgs_nwis,streamflow,hourly,...,40.677778,-74.879167,https://api.waterdata.usgs.gov/ogcapi/v0/colle...,2025-12-02,EST,None,None,None,3993.0,1993.0
1,01396500,South Branch Raritan River near High Bridge NJ,stream gauge,USGS,NJ,Daily average streamflow,m3/s,usgs_nwis,streamflow,daily,...,40.677778,-74.879167,https://api.waterdata.usgs.gov/ogcapi/v0/colle...,2025-12-02,EST,None,None,None,3993.0,1993.0
2,01396660,Mulhockaway Creek at Van Syckel NJ,stream gauge,USGS,NJ,Hourly average streamflow,m3/s,usgs_nwis,streamflow,hourly,...,40.647500,-74.968889,https://api.waterdata.usgs.gov/ogcapi/v0/colle...,2025-12-02,EST,None,None,None,3987.0,1985.0
3,01396660,Mulhockaway Creek at Van Syckel NJ,stream gauge,USGS,NJ,Daily average streamflow,m3/s,usgs_nwis,streamflow,daily,...,40.647500,-74.968889,https://api.waterdata.usgs.gov/ogcapi/v0/colle...,2025-12-02,EST,None,None,None,3987.0,1985.0
4,01396800,Spruce Run at Clinton NJ,stream gauge,USGS,NJ,Hourly average streamflow,m3/s,usgs_nwis,streamflow,hourly,...,40.640000,-74.915556,https://api.waterdata.usgs.gov/ogcapi/v0/colle...,2025-12-02,EST,None,None,None,3991.0,1985.0


Filter our data to include only "hourly" observations to align with the resolution of NWM predictions.

In [219]:
obs_sites = obs_sites[obs_sites.temporal_resolution == 'daily']

Identify all of the gauges that exist in our region of interest.

In [220]:
usgs_gauges = obs_sites.site_id.unique()
usgs_gauges

array(['01396500', '01396660', '01396800', '01397000', '01398000',
       '01398500', '01399100', '01399500', '01399670', '01400000',
       '01400500', '01401650', '01402000', '01402630', '01403060',
       '01403150', '01403400', '01403540', '01403900', '01405030',
       '01405400', '01406050'], dtype=object)

Collect NWM retrospective v3.0 streamflow data for each of these gauges.

In [221]:
# Path to NWM channel routing output
s3_url = 's3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/chrtout.zarr'

Connect to the public store of NWM data. This is a very large dataset and as a result, it's stored using the Zarr format to enable lazy loading and buffered data reading.

In [222]:

%%time 
ds = xr.open_zarr(
    store=s3_url,
    consolidated=True,
    storage_options={
        "anon": True,
        "client_kwargs": {"region_name": "us-east-1"}
    }
)
ds

CPU times: user 1.65 s, sys: 118 ms, total: 1.76 s
Wall time: 4.9 s


<xarray.Dataset> Size: 51TB
Dimensions:         (time: 385704, feature_id: 2776734)
Coordinates:
  * time            (time) datetime64[ns] 3MB 1979-02-01T01:00:00 ... 2023-02-01
  * feature_id      (feature_id) int64 22MB 101 179 ... 1180001803 1180001804
    elevation       (feature_id) float32 11MB dask.array<chunksize=(2776734,), meta=np.ndarray>
    gage_id         (feature_id) |S15 42MB dask.array<chunksize=(2776734,), meta=np.ndarray>
    latitude        (feature_id) float32 11MB dask.array<chunksize=(2776734,), meta=np.ndarray>
    longitude       (feature_id) float32 11MB dask.array<chunksize=(2776734,), meta=np.ndarray>
    order           (feature_id) int32 11MB dask.array<chunksize=(2776734,), meta=np.ndarray>
Data variables:
    crs             |S1 1B ...
    qBtmVertRunoff  (time, feature_id) float64 9TB dask.array<chunksize=(672, 30000), meta=np.ndarray>
    qBucket         (time, feature_id) float64 9TB dask.array<chunksize=(672, 30000), meta=np.ndarray>
    qSfcLatRunoff   (time, feature_id) float64 9TB dask.array<chunksize=(672, 30000), meta=np.ndarray>
    q_lateral       (time, feature_id) float64 9TB dask.array<chunksize=(672, 30000), meta=np.ndarray>
    streamflow      (time, feature_id) float64 9TB dask.array<chunksize=(672, 30000), meta=np.ndarray>
    velocity        (time, feature_id) float64 9TB dask.array<chunksize=(672, 30000), meta=np.ndarray>
Attributes:
    NCO:                  netCDF Operators version 5.1.4 (Homepage = http://n...
    TITLE:                OUTPUT FROM WRF-Hydro v5.3.0-alpha1
    code_version:         v5.3.0-alpha1
    featureType:          timeSeries
    history:              Thu Sep 28 07:58:36 2023: ncatted -O -a missing_val...
    model_configuration:  retrospective
    proj4:                +proj=lcc +units=m +a=6370000.0 +b=6370000.0 +lat_1...

Select data for the gauges identified above. The command below will give us an array of True and False, where "True" corresponds with the `usgs_gauges` identified above and "False" everywhere else. First we need to format the values in our `usgs_gauges` list into the format used in the Zarr store, i.e. 15 character strings.

In [223]:
formatted_usgs_gauges = [f"{gauge_id:>15}".encode('ascii') for gauge_id in usgs_gauges]
print(f'The first gauge now looks like this: {formatted_usgs_gauges[0]}')

The first gauge now looks like this: b'       01396500'


In [224]:
mask = ds.gage_id.isin(formatted_usgs_gauges)

Use this mask to isolate the data within the Zarr store for the gauge locations.

In [225]:
res = ds.where(mask.compute(), drop=True)
res

<xarray.Dataset> Size: 410MB
Dimensions:         (feature_id: 22, time: 385704)
Coordinates:
  * feature_id      (feature_id) int64 176B 9512912 9512948 ... 9515546 9515956
    elevation       (feature_id) float32 88B 62.4 30.74 86.29 ... 11.36 4.41
    gage_id         (feature_id) |S15 330B b'       01403400' ... b'       01...
    latitude        (feature_id) float32 88B 40.66 40.65 40.65 ... 40.4 40.39
    longitude       (feature_id) float32 88B -74.4 -74.68 ... -74.34 -74.39
    order           (feature_id) int32 88B 2 2 3 2 3 1 3 6 2 ... 5 3 3 5 2 3 3 3
  * time            (time) datetime64[ns] 3MB 1979-02-01T01:00:00 ... 2023-02-01
Data variables:
    crs             (feature_id) object 176B b'' b'' b'' b'' ... b'' b'' b'' b''
    qBtmVertRunoff  (time, feature_id) float64 68MB dask.array<chunksize=(672, 22), meta=np.ndarray>
    qBucket         (time, feature_id) float64 68MB dask.array<chunksize=(672, 22), meta=np.ndarray>
    qSfcLatRunoff   (time, feature_id) float64 68MB dask.array<chunksize=(672, 22), meta=np.ndarray>
    q_lateral       (time, feature_id) float64 68MB dask.array<chunksize=(672, 22), meta=np.ndarray>
    streamflow      (time, feature_id) float64 68MB dask.array<chunksize=(672, 22), meta=np.ndarray>
    velocity        (time, feature_id) float64 68MB dask.array<chunksize=(672, 22), meta=np.ndarray>
Attributes:
    NCO:                  netCDF Operators version 5.1.4 (Homepage = http://n...
    TITLE:                OUTPUT FROM WRF-Hydro v5.3.0-alpha1
    code_version:         v5.3.0-alpha1
    featureType:          timeSeries
    history:              Thu Sep 28 07:58:36 2023: ncatted -O -a missing_val...
    model_configuration:  retrospective
    proj4:                +proj=lcc +units=m +a=6370000.0 +b=6370000.0 +lat_1...

To use these data in the evaluation framework we need to put them into the expected data format. The evaluation library is expecting a `pandas.DataFrame` structured in the way:

|date|Variable 1| Variable 2| ... | Variable N|
|---|---|---|---|---|
|2018-01-01| 0.0 | 0.3 | ... | 1.1 |
|2018-01-02| 0.1 | 0.2 | ... | 1.0 |
|...| ... | ... | ... | ... |
|2020-01-01| 0.3 | 0.0 | ... | 1.3 |

The first step is to get the modeled NWM results from the Dask `DataSet` above into a Pandas `DataFrame`.

In [226]:
start_date = datetime(2020,1,1)
end_date = datetime(2022,1,1)

In [227]:
nwm_df = res.drop_vars(['elevation', 'latitude', 'longitude','order', 'feature_id']) \
           .sel(time=slice(start_date, end_date)) \
           .streamflow.to_dataframe() \
           .reset_index()

In [228]:
# format gage_ids from bytestrings to str objects
nwm_df.gage_id = nwm_df.gage_id.str.decode('utf-8').str.strip()

# add "nwm" suffix to gage_id column. This is so that our
# column names are more descriptive after we pivot the data.
nwm_df.gage_id = nwm_df.gage_id + '_nwm'

nwm_df

,time,feature_id,gage_id,streamflow
0,2020-01-01,0,01403400_nwm,0.59
1,2020-01-01,1,01399100_nwm,0.22
2,2020-01-01,2,01396660_nwm,0.73
3,2020-01-01,3,01403540_nwm,0.26
4,2020-01-01,4,01399670_nwm,0.79
...,...,...,...,...
385985,2022-01-01,17,01402000_nwm,8.37
385986,2022-01-01,18,01401650_nwm,0.14
385987,2022-01-01,19,01398000_nwm,0.88
385988,2022-01-01,20,01406050_nwm,0.19


In [229]:
# rename time column to 'date' to be consistent with the format 
# expected by the evaluation framework
nwm_df.rename(columns={'time': 'date'}, inplace=True) 

# pivot the data into the form expected by the evaluation framework
nwm_df = pd.pivot(nwm_df, values='streamflow', index='date', columns='gage_id') 

# remove the 'name' label on the columns axis to make things cleaner.
nwm_df = nwm_df.rename_axis(None, axis=1)

# reset the index
nwm_df.reset_index(inplace=True)

nwm_df

,date,01396500_nwm,01396660_nwm,01396800_nwm,01397000_nwm,01398000_nwm,01398500_nwm,01399100_nwm,01399500_nwm,01399670_nwm,...,01402000_nwm,01402630_nwm,01403060_nwm,01403150_nwm,01403400_nwm,01403540_nwm,01403900_nwm,01405030_nwm,01405400_nwm,01406050_nwm
0,2020-01-01 00:00:00,6.59,0.73,1.69,17.18,1.03,3.19,0.22,4.11,0.79,...,34.529999,0.72,113.429997,0.30,0.59,0.26,6.00,6.01,3.47,1.78
1,2020-01-01 01:00:00,6.54,1.19,1.69,16.91,1.01,3.05,0.32,4.20,1.12,...,34.519999,0.74,112.369997,0.29,0.65,0.26,5.95,5.86,3.36,1.69
2,2020-01-01 02:00:00,6.48,1.04,1.69,16.48,1.01,3.13,0.38,4.29,1.54,...,34.569999,0.80,111.759998,0.35,0.90,0.33,5.99,5.85,3.58,1.67
3,2020-01-01 03:00:00,6.26,0.88,1.69,16.08,1.00,3.38,0.33,4.01,1.51,...,34.599999,0.82,111.249998,0.31,0.75,0.30,5.95,5.71,3.72,1.75
4,2020-01-01 04:00:00,6.13,0.84,1.69,15.60,0.99,3.12,0.28,3.83,1.27,...,34.569999,0.85,110.959998,0.29,0.65,0.26,5.93,5.60,3.77,1.76
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17540,2021-12-31 20:00:00,2.51,0.41,1.72,6.05,0.90,1.11,0.13,1.59,0.46,...,8.200000,0.27,37.009999,0.00,0.10,0.00,2.37,1.49,0.66,0.28
17541,2021-12-31 21:00:00,2.49,0.40,1.72,6.00,0.89,1.05,0.13,1.53,0.44,...,8.240000,0.26,36.709999,0.00,0.10,0.00,2.36,1.46,0.62,0.25
17542,2021-12-31 22:00:00,2.48,0.42,1.72,5.94,0.89,1.10,0.12,1.49,0.44,...,8.280000,0.26,36.409999,0.00,0.10,0.00,2.34,1.43,0.60,0.23
17543,2021-12-31 23:00:00,2.53,0.42,1.72,5.89,0.89,1.15,0.12,1.47,0.45,...,8.330000,0.26,36.159999,0.00,0.10,0.00,2.33,1.41,0.61,0.21


Get observation for each of the sites we identified at the beginning of the notebook. These should still be stored in the `usgs_gauges` variable.

In [230]:
usgs_gauges

array(['01396500', '01396660', '01396800', '01397000', '01398000',
       '01398500', '01399100', '01399500', '01399670', '01400000',
       '01400500', '01401650', '01402000', '01402630', '01403060',
       '01403150', '01403400', '01403540', '01403900', '01405030',
       '01405400', '01406050'], dtype=object)

In [231]:
# use get_point_data to retrieve observation data for each of the usgs_gauge ids
obs_df = hf.get_point_data(dataset="usgs_nwis",
                            variable="streamflow",
                            temporal_resolution="hourly",
                            aggregation='mean',
                            date_start=start_date,
                            date_end=end_date,
                            site_ids= list(usgs_gauges))

In [232]:
# add the 'obs' suffix to the observation columns so that these
# can easily be distinguished from the 'nwm' data 

obs_df = obs_df.set_index('date') \
            .add_suffix('_obs') \
            .reset_index()

Convert the date column to a datetime[64]

In [233]:
obs_df.date = pd.to_datetime(obs_df.date)

In [234]:
obs_df.head()

,date,01396500_obs,01396660_obs,01396800_obs,01397000_obs,01398000_obs,01398500_obs,01399100_obs,01399500_obs,01399670_obs,...,01402000_obs,01402630_obs,01403060_obs,01403150_obs,01403400_obs,01403540_obs,01403900_obs,01405030_obs,01405400_obs,01406050_obs
0,2020-01-01 00:00:00,5.935925,0.749950,1.68102,11.489800,2.189005,1.995150,0.60562,2.831415,0.778250,...,30.351750,0.815040,83.55575,0.078957,0.413180,0.37356,3.615325,3.424300,4.910050,1.426320
1,2020-01-01 01:00:00,6.190625,0.749950,1.68102,11.376600,2.150092,1.995150,0.60562,2.796747,0.768345,...,29.856500,0.798060,81.36250,0.077471,0.399030,0.37356,3.530425,3.254500,4.740250,1.426320
2,2020-01-01 02:00:00,6.126950,0.749950,1.68102,11.157275,2.122500,1.995150,0.59430,2.747930,0.743583,...,29.290500,0.769760,79.52300,0.073509,0.393370,0.36224,3.424300,3.183750,4.584600,1.313120
3,2020-01-01 03:00:00,5.957150,0.734857,1.68102,11.065300,2.072975,1.970387,0.58298,2.747930,0.738630,...,28.724500,0.740045,78.17875,0.073509,0.380635,0.36224,3.318175,3.042250,4.407725,1.313120
4,2020-01-01 04:00:00,5.744900,0.727310,1.68102,10.895500,2.024865,1.945625,0.58298,2.731657,0.718820,...,28.151425,0.725187,76.19775,0.073014,0.373560,0.35092,3.254500,3.006875,4.245000,1.230342


Combine the NWM and Observation DataFrames

In [247]:
df = nwm_df.merge(obs_df, left_on='date', right_on='date', how='inner')

Perform evaluation on each site.

In [275]:
# Import the Evaluation library from the project root.
import numpy as np
import sys
from pathlib import Path

sys.path.append(str((Path.cwd().absolute() / "../../../src").resolve()))
from cssi_evaluation import evaluation_metrics
from cssi_evaluation.model_evaluation import METRICS_DICT
METRICS_DICT.pop('condon') # remove condon because this take a different input

In [283]:
results = {}
for gauge in usgs_gauges:
    nwm_col = f'{gauge}_nwm'
    obs_col = f'{gauge}_obs'

    # subset the data we're interested in and ignore NaNs
    dat = df[[nwm_col, obs_col]].dropna()

    results[gauge] = {}
    for metric_name, metric_func in METRICS_DICT.items():
        results[gauge][metric_name] =metric_func(np.array(dat[nwm_col].to_list()),
                                                   np.array(dat[obs_col].to_list()))

In [285]:
res = pd.DataFrame(results)
res

,01396500,01396660,01396800,01397000,01398000,01398500,01399100,01399500,01399670,01400000,...,01402000,01402630,01403060,01403150,01403400,01403540,01403900,01405030,01405400,01406050
r2,0.701464,0.743977,-3047.310927,0.842972,-0.070226,0.726468,0.481111,0.574758,0.489203,0.672063,...,0.438885,0.039382,0.556648,0.533727,0.648372,-4.387555,0.490162,0.681006,0.492704,0.632134
spearman_rho,0.821709,0.675686,0.225368,0.742434,0.699687,0.764277,0.744468,0.793716,0.673183,0.853370,...,0.750131,0.652620,0.836625,0.392624,0.692408,0.499195,0.687120,0.733799,0.727674,0.718986
mse,16.003762,0.778322,8.232346,30.031908,82.956008,3.872137,1.574576,3.867313,2.890609,224.833885,...,302.087970,2.578151,2627.108862,0.120404,0.383590,3.080108,24.795513,17.063898,9.196400,1.719973
rmse,4.000470,0.882226,2.869207,5.480138,9.108019,1.967775,1.254821,1.966548,1.700179,14.994462,...,17.380678,1.605662,51.255330,0.346993,0.619347,1.755024,4.979509,4.130847,3.032557,1.311477
bias,-0.000357,-0.040107,0.130895,-0.000491,0.082629,-0.145251,0.164054,-0.143014,-0.041965,-0.106351,...,-0.270390,-0.274977,-0.227250,0.213029,0.106853,1.772751,-0.437300,-0.453647,-0.330800,-0.346301
percent_bias,-0.035697,-4.010744,13.089540,-0.049143,8.262859,-14.525068,16.405412,-14.301411,-4.196506,-10.635145,...,-27.038993,-27.497746,-22.725047,21.302896,10.685336,177.275105,-43.729951,-45.364653,-33.080028,-34.630084
abs_rel_bias,0.000357,0.040107,0.130895,0.000491,0.082629,0.145251,0.164054,0.143014,0.041965,0.106351,...,0.270390,0.274977,0.227250,0.213029,0.106853,1.772751,0.437300,0.453647,0.330800,0.346301
total_difference,-23.820187,-397.740255,3784.428013,-68.022621,2071.202774,-4360.962698,847.844805,-5257.680547,-558.453853,-21889.149480,...,-80281.652723,-2633.750530,-190986.411823,254.230890,419.419720,3106.320688,-32206.727921,-24406.822246,-17073.538105,-6225.678540
pearson_r,0.846207,0.862728,0.076582,0.918884,0.563800,0.859112,0.715554,0.804634,0.706155,0.835664,...,0.783725,0.682412,0.825283,0.875868,0.816797,0.524820,0.748245,0.857637,0.754190,0.828889
nse,0.701464,0.743977,-3047.310927,0.842972,-0.070226,0.726468,0.481111,0.574758,0.489203,0.672063,...,0.438885,0.039382,0.556648,0.533727,0.648372,-4.387555,0.490162,0.681006,0.492704,0.632134
